# pyvista-radiation: a PyVista extension

Use ordinary PyVista meshes and plotters to explore saved antenna fields. No solver or standalone application is needed.

Install `pip install -e '.[notebook]'` from the repository into this notebook's kernel environment. Open this notebook in your existing Jupyter or notebook editor. The default trame backend is interactive. If your frontend cannot serve widgets, set `backend = "static"` below; that produces images rather than an interactive view.

This walkthrough does not run the optional terrain/city propagation model.

In [ ]:
import os

import numpy as np
import pyvista as pv

from pyvista_radiation import from_arrays, load_patch, principal_cut, radiation_mesh

# Change to "static" for a frontend without interactive widget support.
backend = os.environ.get("PYVISTA_RADIATION_NOTEBOOK_BACKEND", "trame")
pv.set_jupyter_backend(backend)
data = load_patch()
data.frequency_hz / 1e9

## Add a pattern to your own plotter

The installed package provides the lazy `radiation` plotter component. It returns a normal PyVista actor. Frequency updates retain its styling and camera. The plotter stays under your control.

In [ ]:
plotter = pv.Plotter(notebook=True)
actor = plotter.radiation.add_pattern(data, index=0, radial_scale="db", floor_db=-30)
plotter.radiation.set_frequency(actor, 1)
plotter.show(jupyter_backend=backend)

## Reuse native PyVista filters

The returned geometry is `pv.PolyData`. A clip acts on the display surface; its geometrical area is not physical radiated power. VTP saves that visualization, while `save(path, data)` saves all original complex fields in NPZ.

In [ ]:
mesh = radiation_mesh(data, 1, radial_scale="db", floor_db=-30)
clipped = mesh.clip(normal="x")
clipped.plot(notebook=True, jupyter_backend=backend, scalars="Relative field (dB)", clim=(-30, 0))

## Extract numerical results without opening an app

Cuts use relative field decibels. Per-frequency normalization compares shapes, not gain or efficiency. The reference dataset is a simulated tutorial antenna, not measurement-validated hardware.

In [ ]:
angles_deg, relative_field_db = principal_cut(data, 1, plane="xz")
print(f"{len(angles_deg)} cut samples; peak {relative_field_db.max():g} dB")
print(mesh.field_data)

## Supply your own arrays

Explicitly declare angles, physical units and source. This analytic short dipole has known zero phase, so its real formula is deliberately cast to complex. Real-only input is otherwise rejected; omitted `e_phi` declares a zero component.

In [ ]:
theta = np.linspace(0, 180, 91)
phi = np.arange(0, 360, 5)
e_theta = np.broadcast_to(np.sin(np.deg2rad(theta))[:, None], (91, 72)).astype(complex)
dipole = from_arrays(2.4e9, theta, phi, e_theta, angle_unit="deg",
                     reference_radius_m=1, source="Analytic short dipole",
                     kind="analytic", solver_version="not applicable")
radiation_mesh(dipole, quantity="directivity").plot(
    notebook=True, jupyter_backend=backend, scalars="Directivity (dBi)")

For CSV import, see `docs/importing.rst`. For a standalone script, run `python examples/pyvista_extension.py`. The optional synthetic terrain/city demonstration is launched with `pyvista-radiation --coverage`; it is separate from these native mesh and plotting APIs.